In [ ]:
!pip install indic-nlp-library
!pip install stopwordsiso
!pip install nltk scikit-learn pandas numpy



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 15.0 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 4.4 MB/s eta 0:00:00


In [ ]:
import os
import json
import pandas as pd
import numpy as np
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords

np.random.seed(0)
pd.set_option('display.max_columns', None)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Load DF

In [ ]:

df = pd.read_csv("/content/drive/MyDrive/Transcripts_CSS/outputs/df_new.csv")
df["transcript"] = df["transcript"].fillna("")

# check what you have
print(f"Total videos: {len(df)}")
print(f"Communities: {df['community'].unique()}")
print(f"Language distribution:\n{df['script_class'].value_counts()}")

Total videos: 9206
Communities: ['transcripts_business' 'transcripts_religion' 'transcripts_comedy'
 'transcripts_lifestyle' 'transcripts_tech' 'transcripts_motivational'
 'transcripts_gaming' 'transcripts_politics']
Language distribution:
script_class
hindi-english (code-mixed)    3964
hindi                         2908
english                       2334
Name: count, dtype: int64


Build stopword list

In [ ]:
from stopwordsiso import stopwords as iso_stopwords

# english stopwords
en_stopwords = set(stopwords.words('english'))

# hindi stopwords from stopwordsiso
hi_stopwords = set(iso_stopwords("hi"))

# add manual Hindi stopwords that are commonly missed
manual_hi_stopwords = {
    # common Hindi function words
    "है", "हैं", "था", "थी", "थे", "हो", "हुआ", "हुई", "हुए",
    "का", "की", "के", "को", "से", "में", "पर", "और", "या",
    "यह", "वह", "इस", "उस", "जो", "तो", "भी", "ही", "न",
    "नहीं", "कि", "एक", "हम", "आप", "मैं", "वो", "इसे",
    "उसे", "उन", "इन", "जब", "तब", "अब", "यहाँ", "वहाँ",
    "कर", "करना", "करने", "किया", "किये", "होना", "होने",
    "लिए", "साथ", "बाद", "पहले", "बहुत", "कुछ", "सब",
    # romanized Hindi function words that WhisperX produces
    "hai", "hain", "tha", "thi", "the", "ho", "hoga", "hogi",
    "ka", "ki", "ke", "ko", "se", "mein", "par", "aur", "ya",
    "yeh", "woh", "is", "us", "jo", "toh", "bhi", "hi", "na",
    "nahi", "nahin", "ek", "hum", "aap", "main", "wo",
    "karna", "kiya", "hona", "liye", "saath", "baad", "pehle"
}

# DO NOT add discourse markers to stopwords
# keep: basically, like, right, you know, matlab, dekho, yaar etc.
# these are exactly what you want LDA to find

all_stopwords = en_stopwords | hi_stopwords | manual_hi_stopwords
print(f"Total stopwords: {len(all_stopwords)}")

Total stopwords: 467


Normalisation

In [ ]:
# normalize common romanized Hindi discourse markers and content words
# to a canonical form so LDA doesn't fragment them
normalization_map = {
    # discourse markers — these matter most for GDCF
    "matlab": "मतलब",
    "matalab": "मतलब",
    "dekho": "देखो",
    "dekh": "देख",
    "yaar": "यार",
    "yar": "यार",
    "bhai": "भाई",
    "bhaiya": "भैया",
    "haan": "हाँ",
    "han": "हाँ",
    "accha": "अच्छा",
    "achha": "अच्छा",
    "acha": "अच्छा",
    "sahi": "सही",
    "bilkul": "बिल्कुल",
    "theek": "ठीक",
    "thik": "ठीक",
    "karo": "करो",
    "karo": "करो",
    "baat": "बात",
    "bat": "बात",
    "log": "लोग",
    "paisa": "पैसा",
    "paise": "पैसे",
    "samajh": "समझ",
    "samjho": "समझो",
    "lagta": "लगता",
    "lagti": "लगती",
    # keep English discourse markers as-is — they are valid findings
    "basically": "basically",
    "actually": "actually",
    "like": "like",
    "right": "right",
    "okay": "okay",
    "ok": "okay",
}

def normalize_text(text):
    text = text.lower().strip()
    tokens = text.split()
    normalized = [normalization_map.get(t, t) for t in tokens]
    return " ".join(normalized)

Tokenisation

In [ ]:
import re

def is_valid_token(token):
    """
    Keep a token if:
    - it's a meaningful Devanagari word (length > 1)
    - it's a meaningful Roman word (length > 2, not just punctuation)
    - it's not a number
    - it's not in stopwords
    """
    # remove pure punctuation
    if not re.search(r'[\u0900-\u097F\w]', token):
        return False
    # remove pure numbers
    if re.match(r'^\d+$', token):
        return False
    # remove very short tokens (likely noise)
    if len(token) <= 1:
        return False
    # remove stopwords
    if token in all_stopwords:
        return False
    return True

def tokenize_hinglish(text):
    """
    Tokenize mixed Hindi-English text.
    Simple whitespace tokenizer works better than
    NLTK's word_tokenize for Devanagari.
    """
    # normalize first
    text = normalize_text(text)
    # split on whitespace and punctuation
    tokens = re.split(r'[\s\.,!?;:()\[\]{}"\']+', text)
    # filter
    tokens = [t for t in tokens if is_valid_token(t)]
    return tokens

# apply to all transcripts
print("Tokenizing...")
data_samples = list(df["transcript"])
tokenized_documents = [tokenize_hinglish(doc) for doc in data_samples]
filtered_documents = [" ".join(doc) for doc in tokenized_documents]

# quick sanity check
print(f"\nSample tokens from first transcript:")
print(tokenized_documents[0][:30])
print(f"\nVocabulary size estimate: {len(set(t for doc in tokenized_documents for t in doc))}")

Tokenizing...

Sample tokens from first transcript:
['hey', 'guys', 'size', 'india', 'eyewear', 'market', 'roughly', '$10', 'billion', 'lenskart', 'come', 'ipo', 'priced', '$8', 'billion', 'bubble', '100%', 'yes', 'investor', 'put', 'money', 'zero', 'zero', 'sonata', 'nothing', 'needs', 'put', 'makes', 'sense', 'making']

Vocabulary size estimate: 296231


In [ ]:
def print_top_words(model, feature_names, n_top_words):
    for topic_idx, topic in enumerate(model.components_):
        top_words_idx = topic.argsort()[:-n_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_words_idx]
        print(f"\nTopic {topic_idx+1}:", ", ".join(top_words))

def save_top_words(model, feature_names, n_top_words, community, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, f"LDA_topics_{community}.csv")
    with open(path, "w", encoding="utf-8") as f:
        for topic_idx, topic in enumerate(model.components_):
            top_words_idx = topic.argsort()[:-n_top_words - 1:-1]
            top_words = [feature_names[i] for i in top_words_idx]
            f.write(f"{topic_idx+1},")
            f.write(",".join(top_words))
            f.write("\n")
    print(f"Saved: {path}")

In [ ]:
n_topics = 5

n_top_words = 10
output_dir = "/content/drive/MyDrive/Transcripts_CSS/outputs/LDA_results/"

# ============================================================
# SELECT ONE COMMUNITY
# ============================================================

COMMUNITY = "transcripts_motivational"

community_df = df[df["community"] == COMMUNITY].copy()
community_df = community_df.reset_index(drop=True)

print(f"\n{'='*50}")
print(f"Processing: {COMMUNITY}")
print(f"{'='*50}")

print(f"Videos in community: {len(community_df)}")


# ============================================================
# GET TRANSCRIPTS FOR THIS COMMUNITY
# ============================================================

community_samples = list(
    community_df["transcript"].fillna("")
)

community_tokenized = [
    tokenize_hinglish(doc)
    for doc in community_samples
]

community_filtered = [
    " ".join(doc)
    for doc in community_tokenized
]


# ============================================================
# SKIP IF TOO LITTLE DATA
# ============================================================

if len([
    d for d in community_filtered
    if len(d) > 10
]) < 10:

    raise ValueError(
        f"Not enough data for community: {COMMUNITY}"
    )


# ============================================================
# VECTORIZATION
# ============================================================

count_vectorizer = CountVectorizer(
    min_df=5,        # word must appear in at least 5 videos
    max_df=0.95,     # not in more than 95% of videos
    max_features=10000
)

try:

    X = count_vectorizer.fit_transform(
        community_filtered
    )

    print(
        f"Vocabulary size: {X.shape[1]}"
    )

except Exception as e:

    raise ValueError(
        f"Vectorizer failed for {COMMUNITY}: {e}"
    )


# ============================================================
# FIT LDA
# ============================================================

print(
    f"Fitting LDA ({n_topics} topics)..."
)

lda = LatentDirichletAllocation(
    n_components=n_topics,
    learning_method="online",
    random_state=0,
    max_iter=5,
    evaluate_every=1,
    verbose=1
)

lda.fit(X)


# ============================================================
# PRINT AND SAVE TOP WORDS
# ============================================================

feature_names = (
    count_vectorizer.get_feature_names_out()
)

print_top_words(
    lda,
    feature_names,
    n_top_words
)

save_top_words(
    lda,
    feature_names,
    n_top_words,
    COMMUNITY,
    output_dir
)


# ============================================================
# GET DOCUMENT-TOPIC DISTRIBUTIONS
# ============================================================

doc_topic_probs = lda.transform(X)


# ============================================================
# ADD TOPIC PROBABILITIES TO COMMUNITY DATAFRAME
# ============================================================

topic_cols = pd.DataFrame(
    doc_topic_probs,
    columns=[
        f'Topic_{i+1}_Probability'
        for i in range(n_topics)
    ]
)

community_df = pd.concat(
    [community_df, topic_cols],
    axis=1
)


# ============================================================
# FINAL RESULT
# ============================================================

final_df = community_df

print(
    f"\nDone: {COMMUNITY}"
)

print(
    f"Final df shape: {final_df.shape}"
)


Processing: transcripts_motivational
Videos in community: 1006
Vocabulary size: 6184
Fitting LDA (5 topics)...
iteration: 1 of max_iter: 5, perplexity: 1083.2920
iteration: 2 of max_iter: 5, perplexity: 1044.0059
iteration: 3 of max_iter: 5, perplexity: 1028.5555
iteration: 4 of max_iter: 5, perplexity: 1020.4908
iteration: 5 of max_iter: 5, perplexity: 1015.5184

Topic 1: one, know, like, people, going, want, go, get, time, life

Topic 2: आपक, कर, अगर, शन, हम, आज, बड, लग, टर, बन

Topic 3: आपक, कर, अगर, उसक, बन, हम, आपन, अच, बड, सर

Topic 4: hair, आपक, करतक, करम, बन, हम, कर, शर, print, waiting

Topic 5: relationships, relationship, god, problem, gita, love, mind, krishna, said, thank
Saved: /content/drive/MyDrive/Transcripts_CSS/outputs/LDA_results/LDA_topics_transcripts_motivational.csv

Done: transcripts_motivational
Final df shape: (1006, 16)


In [ ]:
csv_path = "/content/drive/MyDrive/Transcripts_CSS/outputs/LDA_results/final_df/df_LDA_motivational.csv"
final_df.to_csv(csv_path, index=False)
print(f"Saved to: {csv_path}")
display(final_df.head())

Saved to: /content/drive/MyDrive/Transcripts_CSS/outputs/LDA_results/final_df/df_LDA_motivational.csv


,video_id,community,whisperx_language,script_class,hindi_pct,other_pct,english_pct,transcript,transcript_length,duration,file_path,Topic_1_Probability,Topic_2_Probability,Topic_3_Probability,Topic_4_Probability,Topic_5_Probability
0,GaurGopalDas__jgm5axDsbS0.mp3,transcripts_motivational,hi,hindi-english (code-mixed),79.98,0.0,20.02,अनुमान जी एक माचो मेल है। यह एक माचो मेल है। अ...,860,584.461,/content/drive/MyDrive/Transcripts_CSS/transcr...,0.199456,0.745622,0.000815,0.000804,0.053303
1,GaurGopalDas__k-pAIhWB4cY.mp3,transcripts_motivational,en,english,0.00,0.0,100.00,"Namaste and a very, very good evening to all o...",1176,600.038,/content/drive/MyDrive/Transcripts_CSS/transcr...,0.453026,0.000423,0.000424,0.000427,0.545700
2,GaurGopalDas__mVfUqjx8N0c.mp3,transcripts_motivational,en,english,0.00,0.0,100.00,"Very good evening to you, ladies and gentlemen...",1404,599.958,/content/drive/MyDrive/Transcripts_CSS/transcr...,0.918241,0.000348,0.000350,0.000351,0.080710
3,GaurGopalDas__mwlD07T0CYY.mp3,transcripts_motivational,en,english,0.00,0.0,100.00,My sincerest warm greetings to all of you who ...,1521,600.018,/content/drive/MyDrive/Transcripts_CSS/transcr...,0.350722,0.000307,0.000308,0.570518,0.078145
4,GaurGopalDas__nCxpZ_FEYxk.mp3,transcripts_motivational,en,english,0.00,0.0,100.00,Married or unmarried. Monk or not a monk. Man ...,1414,598.937,/content/drive/MyDrive/Transcripts_CSS/transcr...,0.405946,0.000365,0.000366,0.000367,0.592956
